# Stage 3: Post-Hoc Calibration Under External-Dataset Conditions

## Research Objective

Stage 2 evaluated the Stage 1 PneumoniaMNIST classifier on the external RSNA Pneumonia Detection Challenge dataset.

The results showed that the model's probability estimates were not well calibrated on this external dataset. Stage 3 therefore investigates whether **temperature scaling** can improve the reliability of those probability estimates without retraining the classifier.

The classifier itself remains frozen. Only one scalar temperature parameter is learned.

### Experimental design

The saved RSNA predictions are divided into two stratified subsets:

1. **Calibration subset**: used only to learn the temperature.
2. **Held-out evaluation subset**: used only to measure the effect of calibration.

The following metrics are compared before and after temperature scaling:

- Brier score
- Log loss
- Expected Calibration Error (ECE)
- Area Under the Receiver Operating Characteristic curve (AUROC)
- Reliability diagram

The important principle is that the held-out subset is not used when learning the temperature.

## 1. Set up the Stage 3 environment

Stage 3 does not need to read the original RSNA DICOM images again.

Stage 2 already generated the model outputs and saved them in an `.npz` file. Loading those predictions makes the calibration experiment faster and, more importantly, keeps calibration separate from image inference.

The path below is the same prediction path used at the end of the cleaned Stage 2 notebook.

In [ ]:
# Import the libraries used throughout the calibration experiment.
#
# NumPy handles the saved prediction arrays.
# Pandas is used for readable reliability tables.
# Matplotlib is used for the final reliability diagram.
# PyTorch is used to optimise the temperature parameter.
# scikit-learn provides the standard evaluation metrics.

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score
)


print("Libraries loaded successfully.")

## 2. Load the Stage 2 RSNA predictions

The cleaned Stage 2 notebook saves three arrays:

- `logits`: the raw output of the Stage 1 classifier
- `probabilities`: the original model probabilities
- `labels`: the RSNA ground-truth labels

Temperature scaling is naturally applied to the stored logits, so we keep those rather than reconstructing them from probabilities.

In [ ]:
# This is the prediction file created by Stage 2.
#
# The raw RSNA DICOM files are not needed for Stage 3.
prediction_path = Path(
    r"D:\0001_Kings_AI_Project\trustworthy-ai-medical-imaging"
    r"\research_data\rsna_stage2_predictions.npz"
)


# Stop immediately with a clear message if Stage 2 has not
# produced the prediction file yet.
if not prediction_path.exists():
    raise FileNotFoundError(
        "The Stage 2 prediction file was not found.\n\n"
        f"Expected path:\n{prediction_path}\n\n"
        "Complete Stage 2 external inference first."
    )


# Load the arrays saved by Stage 2.
saved_predictions = np.load(
    prediction_path
)


all_logits = saved_predictions["logits"]
all_probabilities = saved_predictions["probabilities"]
all_labels = saved_predictions["labels"]


print("Stage 2 predictions loaded successfully.")
print("Prediction file:", prediction_path)
print("Number of predictions:", len(all_probabilities))
print("Number of labels:", len(all_labels))

## 3. Validate the Stage 2 prediction file

Before fitting any calibration model, we check that the saved arrays describe the same set of RSNA observations.

This also confirms that the probabilities are valid and that the labels contain the expected binary classes.

In [ ]:
# All three arrays must contain one entry for every evaluated RSNA image.
if not (
    len(all_logits)
    == len(all_probabilities)
    == len(all_labels)
):
    raise ValueError(
        "The Stage 2 logits, probabilities, and labels "
        "do not contain the same number of observations."
    )


# The original model probabilities should always be between 0 and 1.
if (
    np.min(all_probabilities) < 0
    or np.max(all_probabilities) > 1
):
    raise ValueError(
        "The Stage 2 probability array contains a value "
        "outside the valid 0 to 1 range."
    )


# Temperature scaling is a binary-classification experiment,
# so we expect only the two classes 0 and 1.
unique_labels = np.unique(all_labels)

if not np.array_equal(
    unique_labels,
    np.array([0, 1])
):
    raise ValueError(
        f"Unexpected label values found: {unique_labels}"
    )


print("Stage 2 prediction file passed validation.")

print("\nPrediction range:")
print(
    f"  Minimum probability: {all_probabilities.min():.6f}"
)
print(
    f"  Maximum probability: {all_probabilities.max():.6f}"
)

print("\nClass distribution:")
print(
    pd.Series(all_labels)
    .value_counts()
    .sort_index()
)

## 4. Create the calibration and held-out evaluation subsets

The full RSNA prediction set is divided into two equal, stratified subsets.

The split is performed on the prediction indices rather than on the images themselves because Stage 2 has already completed inference.

`random_state=42` makes the split reproducible.

In [ ]:
# Create one index for every RSNA prediction.
prediction_indices = np.arange(
    len(all_labels)
)


# Split the observations into two equal groups.
#
# Stratification keeps the proportion of the two RSNA classes
# approximately the same in both groups.
calibration_indices, evaluation_indices = train_test_split(
    prediction_indices,
    test_size=0.50,
    stratify=all_labels,
    random_state=42
)


# Extract the calibration subset.
calibration_logits = all_logits[
    calibration_indices
]

calibration_probabilities = all_probabilities[
    calibration_indices
]

calibration_labels = all_labels[
    calibration_indices
]


# Extract the completely held-out evaluation subset.
evaluation_logits = all_logits[
    evaluation_indices
]

evaluation_probabilities = all_probabilities[
    evaluation_indices
]

evaluation_labels = all_labels[
    evaluation_indices
]


print("Total RSNA predictions:", len(all_labels))

print(
    "Calibration subset:",
    len(calibration_labels)
)

print(
    "Held-out evaluation subset:",
    len(evaluation_labels)
)

print("\nCalibration class distribution:")
print(
    pd.Series(calibration_labels)
    .value_counts()
    .sort_index()
)

print("\nHeld-out evaluation class distribution:")
print(
    pd.Series(evaluation_labels)
    .value_counts()
    .sort_index()
)

## 5. Fit the temperature using the calibration subset only

Temperature scaling changes the confidence of a classifier without changing the classifier's underlying decision function.

For binary classification, the scaled probability is obtained by:

**scaled probability = sigmoid(logit / temperature)**

A temperature greater than 1 makes the logits smaller in magnitude and therefore produces less extreme probabilities.

The temperature is learned by minimising binary cross-entropy on the **calibration subset only**.

The held-out evaluation subset is not used during this optimisation.

In [ ]:
# Convert the calibration logits and labels into PyTorch tensors.
calibration_logits_tensor = torch.tensor(
    calibration_logits,
    dtype=torch.float32
)

calibration_labels_tensor = torch.tensor(
    calibration_labels,
    dtype=torch.float32
)


# We optimise log-temperature rather than temperature itself.
#
# This guarantees that the final temperature is always positive,
# which is required for meaningful temperature scaling.
log_temperature = nn.Parameter(
    torch.zeros(1)
)


# Binary cross-entropy with logits is appropriate because
# temperature scaling operates directly on the raw logits.
loss_function = nn.BCEWithLogitsLoss()


# LBFGS is useful here because there is only one parameter to learn.
optimizer = optim.LBFGS(
    [log_temperature],
    lr=0.01,
    max_iter=100
)


def closure():
    """
    Calculate the calibration loss for the current temperature.

    The optimiser calls this function repeatedly while searching
    for the temperature that minimises negative log-likelihood.
    """

    optimizer.zero_grad()

    # Convert log-temperature into a positive temperature.
    temperature = torch.exp(
        log_temperature
    )

    # Reduce or increase the confidence of the original logits.
    scaled_logits = (
        calibration_logits_tensor
        / temperature
    )

    # Measure how well the scaled logits agree with the known labels.
    loss = loss_function(
        scaled_logits,
        calibration_labels_tensor
    )

    loss.backward()

    return loss


# Learn the temperature using calibration data only.
optimizer.step(
    closure
)


# Convert the learned log-temperature back into the
# temperature used for the final evaluation.
learned_temperature = torch.exp(
    log_temperature
).item()


print(
    f"Learned temperature: {learned_temperature:.4f}"
)

print(
    "Temperature is positive:",
    learned_temperature > 0
)

## 6. Apply the learned temperature to the held-out RSNA subset

The temperature learned above is now applied to observations that were not used during fitting.

This is the key generalisation check for the calibration experiment.

If the temperature improves calibration on this held-out subset, the improvement is less likely to be caused simply by fitting the same observations that are being evaluated.

In [ ]:
# Convert the held-out logits to a PyTorch tensor.
evaluation_logits_tensor = torch.tensor(
    evaluation_logits,
    dtype=torch.float32
)


# Apply the learned temperature to the held-out logits.
scaled_evaluation_logits = (
    evaluation_logits_tensor
    / learned_temperature
)


# Convert the calibrated logits back into probabilities.
calibrated_evaluation_probabilities = torch.sigmoid(
    scaled_evaluation_logits
).numpy()


# Keep the original probabilities for the before/after comparison.
probabilities_before = (
    evaluation_probabilities
)

probabilities_after = (
    calibrated_evaluation_probabilities
)


print("Held-out evaluation completed.")

print("\nOriginal probability range:")
print(
    f"  {probabilities_before.min():.6f}"
    f" to "
    f"{probabilities_before.max():.6f}"
)

print("\nCalibrated probability range:")
print(
    f"  {probabilities_after.min():.6f}"
    f" to "
    f"{probabilities_after.max():.6f}"
)

## 7. Calculate the evaluation metrics

We compare the original and calibrated probabilities on the **same held-out RSNA observations**.

### Brier score

Measures the squared difference between predicted probability and the actual outcome. Lower is better.

### Log loss

Penalises incorrect and overconfident probability estimates. Lower is better.

### Expected Calibration Error

Measures the gap between predicted confidence and observed frequency across probability bins. Lower is better.

### AUROC

Measures ranking/discrimination rather than calibration. It is included to check whether temperature scaling changes the classifier's discrimination.

Temperature scaling should normally change calibration while leaving the ordering of predictions, and therefore AUROC, essentially unchanged.

In [ ]:
# Calculate Expected Calibration Error using the same
# 10-bin approach used in the Stage 2 RSNA analysis.
def calculate_ece(
    y_true,
    probabilities,
    n_bins=10
):
    """
    Calculate Expected Calibration Error.

    The predictions are divided into confidence bins.
    Within each bin, we compare:
        - mean predicted probability
        - observed positive frequency

    Each bin's error is weighted by the number of observations
    it contains.
    """

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    ece = 0.0

    for bin_number in range(n_bins):

        lower = bin_edges[bin_number]
        upper = bin_edges[bin_number + 1]

        # Include the right edge in the final bin so that
        # a probability of exactly 1.0 is not excluded.
        if bin_number == n_bins - 1:

            mask = (
                (probabilities >= lower)
                & (probabilities <= upper)
            )

        else:

            mask = (
                (probabilities >= lower)
                & (probabilities < upper)
            )


        # Empty bins do not contribute to ECE.
        if np.sum(mask) == 0:
            continue


        mean_probability = (
            probabilities[mask].mean()
        )

        observed_frequency = (
            y_true[mask].mean()
        )


        # Weight this bin by the fraction of all observations
        # that fall inside it.
        ece += (
            np.sum(mask) / len(y_true)
        ) * abs(
            mean_probability
            - observed_frequency
        )


    return ece


# Calculate the metrics before temperature scaling.
metrics_before = {
    "Brier": brier_score_loss(
        evaluation_labels,
        probabilities_before
    ),

    "Log Loss": log_loss(
        evaluation_labels,
        probabilities_before
    ),

    "ECE": calculate_ece(
        evaluation_labels,
        probabilities_before
    ),

    "AUROC": roc_auc_score(
        evaluation_labels,
        probabilities_before
    )
}


# Calculate the same metrics after temperature scaling.
metrics_after = {
    "Brier": brier_score_loss(
        evaluation_labels,
        probabilities_after
    ),

    "Log Loss": log_loss(
        evaluation_labels,
        probabilities_after
    ),

    "ECE": calculate_ece(
        evaluation_labels,
        probabilities_after
    ),

    "AUROC": roc_auc_score(
        evaluation_labels,
        probabilities_after
    )
}


# Present the comparison in a compact table.
metrics_table = pd.DataFrame({
    "Before temperature scaling": metrics_before,
    "After temperature scaling": metrics_after
})

metrics_table["Change"] = (
    metrics_table["After temperature scaling"]
    - metrics_table["Before temperature scaling"]
)


print("Held-out RSNA calibration results")
print("=" * 65)

print(
    metrics_table.round(4)
)

## 8. Build reliability tables

A reliability table shows how predicted confidence compares with the actual positive frequency.

For example, if a confidence bin contains predictions averaging 0.70, a well-calibrated model should have an observed positive frequency close to 0.70.

These tables are used to create the final reliability diagram.

In [ ]:
def create_reliability_table(
    y_true,
    probabilities,
    n_bins=10
):
    """
    Create a reliability table for one set of predictions.

    Each row represents one probability-confidence bin.
    """

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    rows = []

    for bin_number in range(n_bins):

        lower = bin_edges[bin_number]
        upper = bin_edges[bin_number + 1]

        if bin_number == n_bins - 1:

            mask = (
                (probabilities >= lower)
                & (probabilities <= upper)
            )

        else:

            mask = (
                (probabilities >= lower)
                & (probabilities < upper)
            )


        count = int(
            np.sum(mask)
        )


        # Leave empty bins in the table so the final diagram
        # retains the full 0 to 1 confidence range.
        if count == 0:

            rows.append({
                "bin_lower": lower,
                "bin_upper": upper,
                "count": 0,
                "mean_probability": np.nan,
                "observed_frequency": np.nan
            })

            continue


        rows.append({
            "bin_lower": lower,
            "bin_upper": upper,
            "count": count,
            "mean_probability": probabilities[mask].mean(),
            "observed_frequency": y_true[mask].mean()
        })


    return pd.DataFrame(rows)


# Reliability before calibration.
reliability_before = create_reliability_table(
    evaluation_labels,
    probabilities_before
)


# Reliability after calibration.
reliability_after = create_reliability_table(
    evaluation_labels,
    probabilities_after
)


print("Reliability before temperature scaling:")
print(
    reliability_before.to_string(index=False)
)

print("\nReliability after temperature scaling:")
print(
    reliability_after.to_string(index=False)
)

## 9. Plot the reliability diagram

The diagonal line represents perfect calibration.

- A point below the diagonal means the model is more confident than the observed outcome frequency supports.
- A point above the diagonal means the model is less confident than the observed outcome frequency.

The comparison before and after temperature scaling gives a visual check of the numerical Brier, log loss, and ECE results.

In [ ]:
# Create the reliability diagram.
plt.figure(
    figsize=(8, 7)
)


# Perfect calibration reference line.
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)


# Reliability before calibration.
plt.plot(
    reliability_before["mean_probability"],
    reliability_before["observed_frequency"],
    marker="o",
    label="Before temperature scaling"
)


# Reliability after calibration.
plt.plot(
    reliability_after["mean_probability"],
    reliability_after["observed_frequency"],
    marker="o",
    label="After temperature scaling"
)


plt.xlabel(
    "Mean predicted probability"
)

plt.ylabel(
    "Observed positive frequency"
)

plt.title(
    "RSNA held-out reliability: "
    "before vs after temperature scaling"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

## 10. Final Stage 3 summary

This final cell records the main calibration result in one place.

The interpretation should focus on the held-out RSNA subset rather than the calibration subset used to learn the temperature.

A successful calibration result is indicated by lower Brier score, lower log loss, and lower ECE, while AUROC should remain approximately unchanged.

In [ ]:
print("STAGE 3 FINAL SUMMARY")
print("=" * 65)

print(
    f"Total RSNA predictions: "
    f"{len(all_labels):,}"
)

print(
    f"Calibration observations: "
    f"{len(calibration_labels):,}"
)

print(
    f"Held-out evaluation observations: "
    f"{len(evaluation_labels):,}"
)

print(
    f"\nLearned temperature: "
    f"{learned_temperature:.4f}"
)


print("\nHeld-out metrics:")

for metric_name in metrics_before:

    before = metrics_before[metric_name]
    after = metrics_after[metric_name]
    change = after - before

    print(
        f"  {metric_name:8s}: "
        f"{before:.4f} -> {after:.4f} "
        f"(change {change:+.4f})"
    )


print("\nStage 3 completed.")
print(
    "The classifier was not retrained. "
    "Only the post-hoc temperature parameter was learned."
)